# Tiny LLM Decoder 2 (Pytorch Implementation Line by Line)

In the second version we are converting everything into Pytorch so that we can perform back propagation using Pytorch.

In [1]:
import numpy as np
import scipy.stats as stats
import torch
import torch.nn as nn
from torch.nn import functional as F

## LLM Architecture

At high level, the training process is as follows:

```text

Text_processing -> Transformer Network -> Compare Loss -> Back Propagation
```

The inference process is as follows:

```text
Text_processing -> Transformer Network -> Predict Next Token -> Repeat until token size reached
```

### Corpus

In [2]:
# Text corpus
corpus = """
mary had a little lamb
its fleece was white as snow
and everywhere that mary went
the lamb was sure to go
mary had a little lamb
the lamb was sure to go
"""


### Parameters

In [3]:
# Set Parameters

# d_model is embedding dimension
d_model = 4

# block_size is the same as context window or maximum sequence length
block_size = 5


## Text Processing

### Setup

In [4]:
# Tokenization

# Split corpus into words
word_tokens = corpus.split()

# Use set to isolate unique words
vocab = sorted(list(set(word_tokens)))
vocab_size = len(vocab)

# Create token id and create lookup table from word to token id
# i is the index or tken id in this case, so we assign token id using sequential number
words_to_id_ref = {word: i for i, word in enumerate(vocab)}

# Create lookup table from id to words
id_to_words_ref = {i: word for i, word in enumerate(vocab)}

In [5]:
vocab_size

19

In [6]:
# convert corpus to id
corpus_in_id = [words_to_id_ref[word] for word in word_tokens]


In [7]:
# Setup embeddings initialized randomly
random_seed = 42
embeddings = torch.nn.Embedding(vocab_size, d_model)
embeddings


Embedding(19, 4)

In [8]:
embeddings.weight

Parameter containing:
tensor([[-0.4537, -0.3037, -0.5820, -0.5554],
        [ 0.0765,  0.6750, -2.5382, -2.2228],
        [ 1.3133,  0.3890, -0.7044,  1.8294],
        [-1.1095, -0.2805, -2.8037, -0.2135],
        [ 0.5842, -0.0190, -0.1686, -0.5690],
        [ 1.1289,  0.6673, -1.8159,  0.2044],
        [ 0.1164,  0.0541, -0.0681, -0.8721],
        [ 0.1110, -0.0263,  1.7564,  0.2022],
        [-0.4216, -0.1478, -1.7813,  0.2273],
        [-0.1116, -2.5532,  0.4795, -0.2581],
        [-0.0345, -0.6270,  0.3185, -0.7964],
        [ 0.3993, -2.3295,  0.3358,  1.4811],
        [-1.2078, -0.8902,  2.1466,  0.9272],
        [-0.3708,  1.5762, -0.9877, -1.3561],
        [-1.0854, -2.4093, -0.0404,  0.9256],
        [ 0.7741, -1.5465,  1.1164,  0.7309],
        [ 0.7890, -1.3984,  1.0840, -0.6327],
        [ 0.5908,  1.0355, -1.1697, -0.4077],
        [-0.0770, -0.4461, -1.4163, -0.6178]], requires_grad=True)

### Text Processing for Training

In [9]:
def get_sequence_block(block_size, start_position, corpus_in_id):
  """
  This function extract the block in id from the corpus in id
  User need to supply the block size and the start position
  Return: X, Y, next_start_position
  """
  X = corpus_in_id[start_position: start_position+block_size]
  Y = corpus_in_id[start_position+1: start_position+block_size+1]
  next_start_position = start_position + block_size
  return X, Y, next_start_position

In [10]:
def get_N(corpus_in_id, block_size):
  """
  This function calculate the number of block we need to run for the entire corpus
  """
  N = (len(corpus_in_id)-1) // block_size
  return N

In [11]:
N = get_N(corpus_in_id, block_size=block_size)
N

6

In [12]:
X, Y, next_start = get_sequence_block(block_size=block_size, start_position=0, corpus_in_id=corpus_in_id)
X, Y, next_start

([10, 6, 0, 9, 8], [6, 0, 9, 8, 7], 5)

In [13]:
# Get embedding based on X token ID
X_input = embeddings(torch.tensor(X, dtype=torch.long))
X_input

tensor([[-0.0345, -0.6270,  0.3185, -0.7964],
        [ 0.1164,  0.0541, -0.0681, -0.8721],
        [-0.4537, -0.3037, -0.5820, -0.5554],
        [-0.1116, -2.5532,  0.4795, -0.2581],
        [-0.4216, -0.1478, -1.7813,  0.2273]], grad_fn=<EmbeddingBackward0>)

In [14]:
# Get positional Endccoding
position = np.arange(0,block_size)
position = position.reshape(-1,1)
position = torch.Tensor(position)
position

tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.]])

In [15]:
X_input_pos = X_input + position
X_input_pos

tensor([[-0.0345, -0.6270,  0.3185, -0.7964],
        [ 1.1164,  1.0541,  0.9319,  0.1279],
        [ 1.5463,  1.6963,  1.4180,  1.4446],
        [ 2.8884,  0.4468,  3.4795,  2.7419],
        [ 3.5784,  3.8522,  2.2187,  4.2273]], grad_fn=<AddBackward0>)

In [16]:
X_input_pos.shape

torch.Size([5, 4])

In [17]:
print(X_input_pos.requires_grad)
print(X_input_pos.shape)

True
torch.Size([5, 4])


The X_input is ready to feed into the transformer network. Lets assign it to X for simplicity.

In [18]:
X = X_input_pos

## Transformer

### Attention Formula

We will be implementing the attention formula so we need to understand it

$$\text{Attention (Q,K,V)} = \text{Softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) V$$


- $Q$ refers to the token we are asking or checking, in our first block `mary` will be the Q.

- $K$ refers to all other words in the block. In our context it will be `mary had a little lamb`. We will need to check `mary` against `mary`.

- $V$ refers to the value.

- $QK$ we multiple $Q$ by $K$ to get the relative position between $Q$ and each $K$. The $T$ refers to transpose, it is a necessary step in doing matrix multiplication.

- $d_k$ is a normalization factor so that the numbers $QK$ are stable. In our case $d_k$ is the model dimension.




### Simple Architecture

In this architecture, we use the simplest architecture

```text

input text pre-processing (X)
      |
      V
Self attention
      |
      V
Residual layer
      |
      V
Normalized Layer
      |
      V
Feed Forward Network
      |
      V
Residual Layer
      |
      V
Normalized Layer

```

In Feed Forward Network, we have 3 layers

```text
Linear Layer
      |
      V
ReLU Layer
      |
      V
Linear Layer

```

## Self Attention Layer

In [67]:
# Create a random generator with a random seed
rnd = np.random.default_rng(random_seed)


In [68]:
# Initializing weights (4,4) for WQ, WK and WV
WQ = nn.Parameter(torch.Tensor(rnd.standard_normal((d_model, d_model))))
WK = nn.Parameter(torch.Tensor(rnd.standard_normal((d_model, d_model))))
WV = nn.Parameter(torch.Tensor(rnd.standard_normal((d_model, d_model))))

In [69]:
print(WQ.requires_grad)
print(WK.requires_grad)
print(WV.requires_grad)

True
True
True


In [70]:
WQ

Parameter containing:
tensor([[ 0.3047, -1.0400,  0.7505,  0.9406],
        [-1.9510, -1.3022,  0.1278, -0.3162],
        [-0.0168, -0.8530,  0.8794,  0.7778],
        [ 0.0660,  1.1272,  0.4675, -0.8593]], requires_grad=True)

In [71]:
WK

Parameter containing:
tensor([[ 0.3688, -0.9589,  0.8785, -0.0499],
        [-0.1849, -0.6809,  1.2225, -0.1545],
        [-0.4283, -0.3521,  0.5323,  0.3654],
        [ 0.4127,  0.4308,  2.1416, -0.4064]], requires_grad=True)

In [72]:
# Calculate QKV
# Q = X @ WQ
Q = X @ WQ
K = X @ WK
V = X @ WV

In [73]:
Q.shape

torch.Size([5, 4])

In [74]:
# Compute attention score
attention_score = Q @ K.T
attention_score.shape

torch.Size([5, 5])

In [75]:
# Divide by sqrt(d_k)
stable_factor = np.sqrt(d_model)
attention_stable = attention_score / stable_factor
attention_stable.shape

torch.Size([5, 5])

## Causal Masking

In [76]:
causal_mask = np.triu(np.ones(attention_stable.shape), k=1)
causal_mask

array([[0., 1., 1., 1., 1.],
       [0., 0., 1., 1., 1.],
       [0., 0., 0., 1., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0.]])

In [77]:
# actual causal masking
causal_mask = torch.triu(
    torch.full_like(attention_stable, float("-inf")),
    diagonal=1
)
causal_mask.shape

torch.Size([5, 5])

In [78]:
attention_stable

tensor([[  2.4224,   1.8777,   2.3737,   9.4358,  10.8836],
        [  4.6734,   6.5272,  14.6472,  21.4405,  26.6513],
        [  8.3731,   7.8566,  37.3868,  49.5877,  54.5998],
        [ 16.3061,  17.3581,  60.8706,  88.3191, 100.4278],
        [ 19.2601,  21.3307,  66.3376,  99.3942, 114.8598]],
       grad_fn=<DivBackward0>)

In [79]:
attention_stable_mask = attention_stable + causal_mask
attention_stable_mask

tensor([[  2.4224,     -inf,     -inf,     -inf,     -inf],
        [  4.6734,   6.5272,     -inf,     -inf,     -inf],
        [  8.3731,   7.8566,  37.3868,     -inf,     -inf],
        [ 16.3061,  17.3581,  60.8706,  88.3191,     -inf],
        [ 19.2601,  21.3307,  66.3376,  99.3942, 114.8598]],
       grad_fn=<AddBackward0>)

In [80]:
attention_stable_mask.shape

torch.Size([5, 5])

In this 5 by 5 matrix we have

| col | mary | had | a | little | lamb |
| --- | --- | --- | --- | --- | --- |
| mary | 0.17 | -inf | -inf | -inf | -inf |
| had | 1.9 | 6.8 | -inf | -inf | -inf |
| a | 4.5 | 15.3 | 33.67 | -inf | -inf |
| little | 4.75 | 16.8 | 36.53 | 46.7 | -inf |
| lamb | 7.45 | 26.5 | 57.5 | 73.38 | 108.5 |


So `mary` can only see `mary` and it need to predict the next token. The second last row `mary had a little` is given but `lamb` is masked. the model suppose to predict mask in the 4th row.

In [81]:
attention_softmax = torch.softmax(attention_stable_mask, dim=-1)
attention_softmax.shape

torch.Size([5, 5])

In [82]:
attention_softmax

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.3542e-01, 8.6458e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [2.5090e-13, 1.4969e-13, 1.0000e+00, 0.0000e+00, 0.0000e+00],
        [5.3108e-32, 1.5207e-31, 1.2003e-12, 1.0000e+00, 0.0000e+00],
        [3.0310e-42, 2.4034e-41, 8.4543e-22, 1.9204e-07, 1.0000e+00]],
       grad_fn=<SoftmaxBackward0>)

In [83]:
attention_output = attention_softmax @ V
attention_output.shape

torch.Size([5, 4])

In [84]:
attention_output

tensor([[-1.8476, -3.4518, -0.2452,  2.3517],
        [-1.9115, -2.8676,  1.8532,  3.2429],
        [ 1.5197, -0.6816,  0.2991,  4.3915],
        [ 3.2685, -0.7611, -3.1683,  6.4162],
        [ 0.2491, -4.8184, -0.9857,  9.6613]], grad_fn=<MmBackward0>)

### Residual Layer

In [85]:
attention_output_residual = attention_output + X
attention_output_residual.shape

torch.Size([5, 4])

### Normalized Layer

In [86]:
norm1 = nn.LayerNorm(d_model)
attention_output_residual_scaled = norm1(attention_output_residual)
attention_output_residual_scaled.shape

torch.Size([5, 4])

### Feed Forward Layer (FFN)

In [87]:
ffn = nn.Sequential(
    nn.Linear(d_model, d_model, bias=False),
    nn.ReLU(),
    nn.Linear(d_model, d_model, bias=False)
)

In [88]:
# FFN 1 - Linear Layer
ffn_1 = ffn(attention_output_residual_scaled)
ffn_1.shape

torch.Size([5, 4])

### Second Residual Layer

In [89]:
seond_residual = ffn_1 + attention_output_residual_scaled
seond_residual.shape

torch.Size([5, 4])

### Second Normalized Layer

In [90]:
# 2nd layer
norm2 = nn.LayerNorm(d_model)
second_norm = norm2(seond_residual)
second_norm.shape

torch.Size([5, 4])

In [91]:
transformer_output = second_norm
transformer_output.shape

torch.Size([5, 4])

### Convert to Logits

In [92]:
embeddings.weight.shape

torch.Size([19, 4])

In [93]:
logits = transformer_output @ embeddings.weight.T
logits.shape

torch.Size([5, 19])

In [94]:
logits

tensor([[ 2.1094,  0.0661, -1.6950,  0.0551, -1.8809, -1.4256, -0.2439,  0.5396,
         -2.8435, -2.5797, -1.1126, -1.3095, -0.2790, -0.0391, -0.8178, -3.4522,
          0.5987, -1.2080,  0.7161],
        [ 1.4990,  0.3507, -0.7065, -2.1010,  0.9252,  0.7755,  0.4851,  2.3878,
         -1.9786, -0.6017, -3.6032, -1.1858, -2.0023, -1.0355,  0.3530, -6.5596,
          0.0653, -1.1332,  0.6518],
        [ 2.5068,  0.3735, -1.9859, -0.7103, -0.9144, -1.5760, -1.0953,  0.8769,
         -3.1092, -1.4138, -2.1601, -1.7469, -0.6200, -0.0356, -0.1821, -3.6904,
          0.3883, -1.6185, -0.2164],
        [ 1.7506, -0.0903, -1.3077,  0.1040, -1.9935, -0.8396,  0.5523,  0.7563,
         -2.5494, -3.0667, -0.9810, -1.0071, -0.4528, -0.2833, -1.0381, -4.1595,
          0.6493, -0.9333,  1.4187],
        [ 1.8741,  0.1221, -1.2710, -0.7725, -0.9470, -0.4277,  0.3362,  1.3898,
         -2.5833, -2.1659, -2.1202, -1.2331, -1.0453, -0.5296, -0.4978, -5.1896,
          0.4491, -1.1547,  1.0128]], grad

### Calculate Loss

In [95]:
criterion = nn.CrossEntropyLoss()

## Linking Everything - Forward Pass

In [107]:
def forward(X):
    # Word embeddings + positional encoding
    X_input = embeddings(X)
    position = torch.arange(len(X), dtype=X_input.dtype).reshape(-1, 1)
    X_input_pos = X_input + position

    # Q, K, V projections
    Q = X_input @ WQ
    K = X_input @ WK
    V = X_input @ WV

    # Scaled dot-product attention
    attention_score = Q @ K.T
    stable_factor = np.sqrt(d_model)
    attention_stable = attention_score / stable_factor

    # Causal masking + softmax
    causal_mask = torch.triu(
        torch.full_like(attention_stable, float("-inf")),
        diagonal=1
    )
    attention_stable_mask = attention_stable + causal_mask
    attention_softmax = torch.softmax(attention_stable_mask, dim=-1)

    # Attention output + residual + norm1
    attention_output = attention_softmax @ V
    attention_output_residual = attention_output + X_input
    norm1 = nn.LayerNorm(d_model)
    attention_output_residual_scaled = norm1(attention_output_residual)

    # FFN + residual + norm2
    ffn = nn.Sequential(
        nn.Linear(d_model, d_model, bias=False),
        nn.ReLU(),
        nn.Linear(d_model, d_model, bias=False)
    )
    ffn_1 = ffn(attention_output_residual_scaled)
    seond_residual = ffn_1 + attention_output_residual_scaled
    norm2 = nn.LayerNorm(d_model)
    transformer_output = norm2(seond_residual)

    # Vocabulary projection
    logits = transformer_output @ embeddings.weight.T

    return logits

We include embeddings lookup and positional encoding in the forward loop, thus we need to extract the first block and feed into the forward function.

> During implementation `X_input = embeddings(X)` was used because we already convert to tensor before feeding in to teh function.

> We also change the code `position = torch.arange(len(X), dtype=X_input.dtype).reshape(-1, 1)` to be pytorch friendly.

In [108]:
X, Y, next_start = get_sequence_block(block_size=block_size, start_position=0, corpus_in_id=corpus_in_id)
X = torch.tensor(X, dtype=torch.long)
logits = forward(X)
logits.shape

torch.Size([5, 19])

The code above is in preparation for a training loop with back propagation.

## END